# Write to warehouse

In [10]:
from pathlib import Path
import sys
from pyspark.sql import functions as sf

In [11]:
# Make repo root importable regardless of Jupyter working directory
root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from notebooks.spark_session import create_spark_session

spark = create_spark_session('build-repo-candidates')
spark

In [12]:
print('master:', spark.sparkContext.master)
print('app:', spark.sparkContext.appName)

master: spark://spark-master:7077
app: build-repo-candidates


## Github repository topics

In [17]:
url = "s3a://gba-bronze-zone-prod/github_enrichment/github_repo_topics/dt=2026-04-05/hr=20/1775511776.5012887.aa14acb197.parquet"
df_topics = spark.read.parquet(url)
df_topics.printSchema()

[Stage 0:>                                                          (0 + 1) / 1]

root
 |-- repo_id: long (nullable = true)
 |-- repo_full_name: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- dt: string (nullable = true)
 |-- hr: string (nullable = true)
 |-- _dlt_load_id: string (nullable = true)
 |-- _dlt_id: string (nullable = true)



In [33]:
df_topics.show()

+-------+--------------------+--------------+----------+---+------------------+--------------+
|repo_id|      repo_full_name|         topic|        dt| hr|      _dlt_load_id|       _dlt_id|
+-------+--------------------+--------------+----------+---+------------------+--------------+
|  13966|         infused/dbf|      database|2026-04-05| 20|1775511776.5012887|DARSDs6YLlArcA|
|  13966|         infused/dbf|         dbase|2026-04-05| 20|1775511776.5012887|YUXcNajDCfjVJQ|
|  22601|sporkmonger/addre...|           uri|2026-04-05| 20|1775511776.5012887|5CeE8wIkV/cRww|
|  13966|         infused/dbf|           dbf|2026-04-05| 20|1775511776.5012887|zR8ZZ/dlxcyxoA|
|  36502|             git/git|             c|2026-04-05| 20|1775511776.5012887|Npcp3i0c1+opOg|
|  22601|sporkmonger/addre...|  uri-template|2026-04-05| 20|1775511776.5012887|OUiUEHM5hNVxsA|
|  13966|         infused/dbf|        foxpro|2026-04-05| 20|1775511776.5012887|WzhWfih6xnthyg|
|  85953|gitextensions/git...|           git|2026-

In [32]:
df_topics.select('repo_id').distinct().count()

261

## Github repository snapshot

In [19]:
url_1 = "s3a://gba-bronze-zone-prod/github_enrichment/github_repo_snapshot/dt=2026-04-05/hr=20/1775511776.5012887.f1cc0b5beb.parquet"
df_snapshot = spark.read.parquet(url_1)
df_snapshot.select("*").printSchema()

root
 |-- repo_id: long (nullable = true)
 |-- repo_full_name: string (nullable = true)
 |-- repo_name: string (nullable = true)
 |-- owner_login: string (nullable = true)
 |-- owner_id: long (nullable = true)
 |-- owner_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- language: string (nullable = true)
 |-- license_spdx_id: string (nullable = true)
 |-- default_branch: string (nullable = true)
 |-- visibility: string (nullable = true)
 |-- is_fork: boolean (nullable = true)
 |-- is_archived: boolean (nullable = true)
 |-- is_disabled: boolean (nullable = true)
 |-- stargazers_count: long (nullable = true)
 |-- forks_count: long (nullable = true)
 |-- watchers_count: long (nullable = true)
 |-- subscribers_count: long (nullable = true)
 |-- open_issues_count: long (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- pushed_at: timestamp (nullable = true)
 |-- api_fetched_at: timestamp (nullable = 

In [26]:
df_snapshot.count()

704

In [35]:
df_snapshot.select('repo_id', 'repo_full_name', 'owner_login', 'owner_id', 'description').show()

+-------+--------------------+-------------+--------+--------------------+
|repo_id|      repo_full_name|  owner_login|owner_id|         description|
+-------+--------------------+-------------+--------+--------------------+
|    682|    tobi/delayed_job|         tobi|     347|Database backed a...|
|  13966|         infused/dbf|      infused|    1780|DBF is a small, f...|
|  21716|      yob/pdf-reader|          yob|    8132|The PDF::Reader l...|
|  22601|sporkmonger/addre...|  sporkmonger|    1778|Addressable is an...|
|  36502|             git/git|          git|   18133|Git Source Code M...|
|  85953|gitextensions/git...|gitextensions| 1700077|Git Extensions is...|
| 184460|       celery/celery|       celery|  319983|Distributed Task ...|
| 184981| memcached/memcached|    memcached|   41836|memcached develop...|
| 229738|           jdbi/jdbi|         jdbi| 9668860|The Jdbi library ...|
| 255478| orbeon/orbeon-forms|       orbeon|  105775|Orbeon Forms is a...|
| 279553|             sbt

## Github repo candidates

In [22]:
url_3 = "s3a://gba-bronze-zone-prod/github_enrichment/repo_candidates/dt=2026-04-05/hr=20/1775511776.5012887.8d4ecd09de.parquet"
df_candidates = spark.read.parquet(url_3)
df_candidates.printSchema()

root
 |-- repo_id: long (nullable = true)
 |-- repo_full_name: string (nullable = true)
 |-- total_events: long (nullable = true)
 |-- public_events_count: long (nullable = true)
 |-- unique_actors: long (nullable = true)
 |-- last_event_at: timestamp (nullable = true)
 |-- bot_events: long (nullable = true)
 |-- issues_count: long (nullable = true)
 |-- comments_count: long (nullable = true)
 |-- releases_count: long (nullable = true)
 |-- merged_pr_count: long (nullable = true)
 |-- pull_request_review_events: long (nullable = true)
 |-- push_events: long (nullable = true)
 |-- gollum_events: long (nullable = true)
 |-- release_events: long (nullable = true)
 |-- commit_comment_events: long (nullable = true)
 |-- create_events: long (nullable = true)
 |-- pull_request_review_comment_events: long (nullable = true)
 |-- issue_comment_events: long (nullable = true)
 |-- delete_events: long (nullable = true)
 |-- issues_events: long (nullable = true)
 |-- fork_events: long (nullable = tr

In [27]:
df_candidates.count()

719

In [39]:
df_candidates.join(
    df_snapshot, 
    df_candidates.repo_id == df_snapshot.repo_id,
    "left"
).count()

719

## Github repository selection reasons

In [41]:
url_4 = "s3a://gba-bronze-zone-prod/github_enrichment/org_candidates__selection_reasons/dt=2026-04-05/hr=20/1775511760.5131924.af3c8d3c88.parquet"
df_selection_reasons = spark.read.parquet(url_4)
df_selection_reasons.show()

+--------------------+--------------+-------------+--------------+
|               value|_dlt_parent_id|_dlt_list_idx|       _dlt_id|
+--------------------+--------------+-------------+--------------+
|           bot_ratio|+Cl/YHLbwKnOfw|            0|z8UV9GvRmPiuXQ|
|commit_comment_ev...|+Cl/YHLbwKnOfw|            1|cBxu5TTX0L/ltA|
|commit_comment_ev...|+Cl/YHLbwKnOfw|            2|8AaUzdh5Vnw3Xw|
|   discussion_events|+Cl/YHLbwKnOfw|            3|sgF8VPaB4gA6Mg|
|discussion_events...|+Cl/YHLbwKnOfw|            4|WS6L4kTZemoJSw|
|       gollum_events|+Cl/YHLbwKnOfw|            5|xTuRGi/tXbP+Og|
| gollum_events_ratio|+Cl/YHLbwKnOfw|            6|IPNHI1AOeQk0Jg|
|       member_events|+Cl/YHLbwKnOfw|            7|fpH3l8iMXaC2RA|
| member_events_ratio|+Cl/YHLbwKnOfw|            8|b4lyZ0Vk9P9xEQ|
|     merged_pr_count|+Cl/YHLbwKnOfw|            9|rm8JnHbLu4Ydlw|
|       public_events|+Cl/YHLbwKnOfw|           10|VZ/JnVI0MM5BnQ|
| public_events_ratio|+Cl/YHLbwKnOfw|           11|fMoqPO9fA6k

26/04/07 07:23:36 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 29928957 ms exceeds timeout 120000 ms
26/04/07 07:23:37 ERROR TaskSchedulerImpl: Lost executor 0 on 192.168.117.7: worker lost: Not receiving heartbeat for 60 seconds
26/04/07 07:23:37 ERROR Inbox: Ignoring error
java.lang.AssertionError: assertion failed: BlockManager re-registration shouldn't succeed when the executor is lost
	at scala.Predef$.assert(Predef.scala:223)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:727)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:133)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:103)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:100)
	at org.apache.spark.rpc.netty.MessageLoop.org$ap

In [18]:
url = "s3a://gba-marts-prod/repositories/dt=2026-04-08/hr=0"
df_repos = spark.read.parquet(url)
df_repos.select('repo_id', 'repo_full_name', 'created_at').show()

+-------+--------------------+-------------------+
|repo_id|      repo_full_name|         created_at|
+-------+--------------------+-------------------+
|   3314|         spree/spree|2008-03-10 14:45:35|
|  47476|         qtile/qtile|2008-08-30 00:16:40|
|  69609|internetarchive/o...|2008-10-30 05:20:14|
|  85953|gitextensions/git...|2008-12-06 09:18:59|
|  95658|  datanoise/vimfiles|2008-12-23 04:18:36|
| 117646|rubygems/rubygems...|2009-01-29 22:30:46|
| 206444|         apache/hive|2009-05-21 02:31:01|
| 297058|inaturalist/inatu...|2009-09-04 02:26:52|
| 302322|       gradle/gradle|2009-09-09 18:27:19|
| 304743|         gbdev/rgbds|2009-09-12 03:33:35|
| 332995| dealforest/dotfiles|2009-10-10 08:47:45|
| 341270|       XCSoar/XCSoar|2009-10-18 14:15:39|
| 345337|openframeworks/op...|2009-10-21 21:55:54|
| 356066|apache/trafficserver|2009-10-31 08:00:10|
| 374927|          erlang/otp|2009-11-16 17:17:57|
| 437011|         openzfs/zfs|2009-12-14 20:20:34|
| 457410|dcwalker/TildeSla...|2

In [14]:
url = "s3a://gba-marts-prod/organizations/dt=2026-04-08/hr=0"
df_orgs = spark.read.parquet(url)
df_orgs.printSchema()

root
 |-- org_id: long (nullable = true)
 |-- org_login: string (nullable = true)
 |-- total_events: long (nullable = true)
 |-- public_events_count: long (nullable = true)
 |-- unique_actors: long (nullable = true)
 |-- last_event_at: timestamp (nullable = true)
 |-- bot_events: long (nullable = true)
 |-- issues_count: long (nullable = true)
 |-- comments_count: long (nullable = true)
 |-- releases_count: long (nullable = true)
 |-- merged_pr_count: long (nullable = true)
 |-- pull_request_review_events: long (nullable = true)
 |-- push_events: long (nullable = true)
 |-- gollum_events: long (nullable = true)
 |-- release_events: long (nullable = true)
 |-- commit_comment_events: long (nullable = true)
 |-- create_events: long (nullable = true)
 |-- pull_request_review_comment_events: long (nullable = true)
 |-- issue_comment_events: long (nullable = true)
 |-- delete_events: long (nullable = true)
 |-- issues_events: long (nullable = true)
 |-- fork_events: long (nullable = true)
 |

In [19]:
df_repos.createOrReplaceTempView("repo_marts")

In [23]:
spark.sql("""
 select
         repo_id, repo_full_name,
          count(*) as repo_count,
          sum(total_events) as total_events,
          sum(push_events) as total_push_events,
          sum(pull_request_events) as total_pull_request_events,
          avg(composite_score) as avg_composite_score
      from repo_marts
      group by repo_id, repo_full_name
""").show()

[Stage 8:>                                                          (0 + 1) / 1]

+----------+--------------------+----------+------------+-----------------+-------------------------+-------------------+
|   repo_id|      repo_full_name|repo_count|total_events|total_push_events|total_pull_request_events|avg_composite_score|
+----------+--------------------+----------+------------+-----------------+-------------------------+-------------------+
|   4253774|         akrherz/iem|         1|           1|                0|                        0|0.48520302639196167|
|  16347164|     buildkite/agent|         1|           4|                0|                        4| 0.9433483923290392|
| 793947316|      lowcodebox/lib|         1|          39|                5|                        0| 2.0215307016304958|
|1120345915|       rigds/openwrt|         1|           4|                0|                        0| 0.9433483923290392|
| 114330218| ESCOMP/CISM-wrapper|         1|           1|                0|                        0|0.48520302639196167|
|1158202268|schedmaster/

In [6]:
df_orgs.show()

26/04/09 09:09:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+------+-----------+------------+-------------------+-------------+-------------------+----------+------------+--------------+--------------+---------------+--------------------------+-----------+-------------+--------------+---------------------+-------------+----------------------------------+--------------------+-------------+-------------+-----------+-------------+-------------+------------+-------------------+-----------------+-----------+--------------------------------+-------------------+-------------------+--------------------+---------------------------+--------------------+----------------------------------------+--------------------------+--------------------+-------------------+--------------------+-------------------+-------------------+--------------------+-------------------------+-----------------------+-------------------+-------------------+----------+---+--------------------+--------------------+--------------------+--------------------+--------------------+--------

In [7]:
spark.read.parquet("s3a://gba-marts-prod/organizations/dt=2026-04-08/hr=6/").count()

571

In [8]:
df = spark.read.parquet("s3a://gba-marts-prod/organizations/dt=2026-04-08/hr=6/")
df.printSchema()

[Stage 8:>                                                          (0 + 1) / 1]

root
 |-- org_id: long (nullable = true)
 |-- org_login: string (nullable = true)
 |-- total_events: long (nullable = true)
 |-- public_events_count: long (nullable = true)
 |-- unique_actors: long (nullable = true)
 |-- last_event_at: timestamp (nullable = true)
 |-- bot_events: long (nullable = true)
 |-- issues_count: long (nullable = true)
 |-- comments_count: long (nullable = true)
 |-- releases_count: long (nullable = true)
 |-- merged_pr_count: long (nullable = true)
 |-- pull_request_review_events: long (nullable = true)
 |-- push_events: long (nullable = true)
 |-- gollum_events: long (nullable = true)
 |-- release_events: long (nullable = true)
 |-- commit_comment_events: long (nullable = true)
 |-- create_events: long (nullable = true)
 |-- pull_request_review_comment_events: long (nullable = true)
 |-- issue_comment_events: long (nullable = true)
 |-- delete_events: long (nullable = true)
 |-- issues_events: long (nullable = true)
 |-- fork_events: long (nullable = true)
 |

In [24]:
spark.stop()